In [1]:
import tidy3d as td
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib widget
from matplotlib.colors import LogNorm

# import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import gc
import os
api_key = os.environ.get("TIDY3D_API_KEY")
import tidy3d.web as web
# web.configure("_blank_")
web.configure(api_key)

Configured successfully.


In [2]:
web.test()

03:38:42 UTC Authentication configured successfully!

In [3]:
h_planck = 6.62607015e-34
e_charge = 1.602176634e-19
def hz_to_ev(f): return (h_planck * f) / e_charge
def ev_to_hz(E): return (E * e_charge) / h_planck

In [4]:
save_dir = '/app/local_project/24Jan dimer holes'
os.makedirs(save_dir, exist_ok=True)

name = 'dipolEz_Th100D200G50' + '_dimer_IN_slice0'
os.makedirs('field_plots_'+ name, exist_ok=True)

In [5]:
def extract_monitor_data(filepath, monitor_name):
    
    # Load full simulation (unavoidable)
    sim_data = td.SimulationData.from_file(filepath)
    
    # Extract ONLY the monitor we need
    monitor_data = sim_data[monitor_name]
    
    # Delete the full simulation data immediately
    del sim_data
    gc.collect()
    
    return monitor_data

In [6]:
def get_plane_config(plane):
    """Return slicing info, labels, and field pairs for a given plane."""
    configs = {
        'Exy': {'slicer': {'z': 0},
            'U_field': 'Ex', 'V_field': 'Ey',
            'xlabel': 'x (nm)', 'ylabel': 'y (nm)',
            'coord_keys': ('x', 'y')},
        
        'Exz': {'slicer': {'y': 0},
            'U_field': 'Ex', 'V_field': 'Ez',
            'xlabel': 'x (nm)', 'ylabel': 'z (nm)',
            'coord_keys': ('x', 'z')},
        
        'Eyz': {'slicer': {'x': 0},
            'U_field': 'Ey', 'V_field': 'Ez',
            'xlabel': 'y (nm)', 'ylabel': 'z (nm)',
            'coord_keys': ('y', 'z')},
    }
    if plane not in configs:
        raise ValueError(f"plane must be one of {list(configs.keys())}")
    
    return configs[plane]


# ========================================


In [7]:
def get_field_component(monitor_data, monitor_data0, comp, slicer, peak, normalize):
    """Extract field component from monitor data"""
    data = getattr(monitor_data, comp).isel(**slicer).interp(f=peak)
    data0 = getattr(monitor_data0, comp).isel(**slicer).interp(f=peak)
    
    if normalize:
        return (data - data0) / data0
    else:
        return data - data0


def get_coords(monitor_data, plane):
    """Extract and format coordinates for plotting"""
    coord_key1, coord_key2 = get_plane_config(plane)['coord_keys']
    
    # Extract coordinates and convert to nm
    coord1 = monitor_data.Ex.coords[coord_key1].values * 1e3
    coord2 = monitor_data.Ex.coords[coord_key2].values * 1e3
    
    # Ensure increasing order
    if coord1[0] > coord1[-1]:
        coord1 = coord1[::-1]
    if coord2[0] > coord2[-1]:
        coord2 = coord2[::-1]
    
    return coord1, coord2


# ========================================


In [8]:
def prepare_Efield_data(Ex, Ey, Ez, coord1, coord2, plane):
    """Compute |E| and meshgrid from field components"""
    # Convert to real NumPy arrays
    Ex = np.real(np.array(Ex))
    Ey = np.real(np.array(Ey))
    Ez = np.real(np.array(Ez))
    
    # Expected shape
    expected_shape = (len(coord2), len(coord1))
    
    # Transpose if needed
    if Ex.shape != expected_shape:
        Ex, Ey, Ez = Ex.T, Ey.T, Ez.T
    
    # For xy plane, apply additional transpose
    # if plane == 'Exy':
    #     Ex, Ey, Ez = Ex.T, Ey.T, Ez.T
    
    # Compute magnitude
    E = np.sqrt(np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2)
    
    # Build coordinate mesh
    horizontal_axis, vertical_axis = np.meshgrid(coord1, coord2)
    
    return E, horizontal_axis, vertical_axis, Ex, Ey, Ez


# ========================================


In [10]:
def plot_Efield(E, horizontal_axis, vertical_axis, U, V, 
               xlabel, ylabel, plane, peak, name, monitor,
               fixed_scale, density, arrow, save_path=None):

    """Render field magnitude and streamlines"""
    plt.figure(figsize=(8, 6))
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f"{name} | {plane}-{monitor} | {hz_to_ev(peak):.3f} eV")
    
    # Color scale
    if fixed_scale:
        cmap_data = E
        cbar_label = '|E| induced'
        vmin, vmax = np.percentile(E, [1, 99])
    else:
        cmap_data = np.abs(E) * 100
        cbar_label = '|E| (%)'
        vmin, vmax = 0, 1000
    
    # Draw color map
    plt.pcolormesh(horizontal_axis, vertical_axis, cmap_data, 
               cmap='inferno', shading='auto',
               norm=LogNorm(vmin=vmin, vmax=100*vmax))
    print(f"h_axis{horizontal_axis.shape}", f"v_axis{vertical_axis.shape}", f"cmap{cmap_data.shape}")
#     plt.pcolormesh(horizontal_axis, vertical_axis, cmap_data[:-1, :-1],
#                 cmap='inferno', shading='auto',
#                 norm=LogNorm(vmin=vmin, vmax=100*vmax)
# )
#     plt.imshow(
#     cmap_data.T,
#     origin='lower',
#     aspect='auto',
#     extent=[ horizontal_axis.min(), horizontal_axis.max(),
#              vertical_axis.min(), vertical_axis.max()  ],
#     cmap='inferno',
#     norm=LogNorm(vmin=vmin, vmax=100*vmax)
# )

    plt.colorbar(label=cbar_label)
    
    # Streamlines
    plt.streamplot(horizontal_axis, vertical_axis, U, V,
                   density=density,
                   linewidth=(E - E.min()) / (E.max() - E.min()) + 0.05,
                   color='white',
                   arrowstyle=arrow)
    
    plt.gca().set_aspect('equal', adjustable='box')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Saved: {os.path.basename(save_path)}")
    
    plt.close()

In [11]:
def plot_Efield_stream(monitor_data, monitor_data0, monitor, peak, plane='Exz', 
                      normalize=False, fixed_scale=True,
                      density=2.5, arrow='fancy, head_length=0.7',
                      save_path=None):
    """Orchestrate field extraction and plotting for a 2D plane"""
    cfg = get_plane_config(plane)
    
    # Extract field components
    Ex = get_field_component(monitor_data, monitor_data0, 'Ex', cfg['slicer'], peak, normalize)
    Ey = get_field_component(monitor_data, monitor_data0, 'Ey', cfg['slicer'], peak, normalize)
    Ez = get_field_component(monitor_data, monitor_data0, 'Ez', cfg['slicer'], peak, normalize)
    
    # Get coordinates and prepare data
    coord1, coord2 = get_coords(monitor_data, plane)
    E, X, Y, Ex, Ey, Ez = prepare_Efield_data(Ex, Ey, Ez, coord1, coord2, plane)
    
    # Select vector components for streamlines
    field_map = {'Ex': Ex, 'Ey': Ey, 'Ez': Ez}
    U = field_map[cfg['U_field']]
    V = field_map[cfg['V_field']]
    
    # Plot
    plot_Efield(E, X, Y, U, V, cfg['xlabel'], cfg['ylabel'], plane, peak, 
               name, monitor, fixed_scale, density, arrow, save_path=save_path)


In [12]:
def process_single_monitor(monitor_name, peak, plane, save_path):

    # Extract only this monitor from both files
    monitor_data = extract_monitor_data(f'{save_dir}/{name}.hdf5', monitor_name)
    monitor_data0 = extract_monitor_data(f'{save_dir}/{name}_empty.hdf5', monitor_name)
    
    # Process and plot
    plot_Efield_stream(
        monitor_data, monitor_data0,
        monitor=monitor_name,
        peak=peak,
        plane=plane,
        normalize=False,
        save_path=save_path
    )
    
    # Free memory
    del monitor_data, monitor_data0
    gc.collect()

In [13]:
# sim_data        = td.SimulationData.from_file(f'{save_dir}/{name}.hdf5')

# sim_data.simulation.plot_3d()
# print(sim_data.simulation.mediums)
# for monitor in sim_data.simulation.monitors:
#     print(monitor.name, monitor.type)

In [14]:
EV_Efield = [
    # 8.42, 16.38  # D100L100
    # 8.27, 16.65             # Th100 D100 
    # 10.35, 15.28 # D100 L25
    # 10.19, 5.35, 20.57# D200L100
    5.051, 10.1376           #Th100 D200
    # 10.24,           # D200 L25
]
for EV in EV_Efield:
    process_single_monitor(
            monitor_name='DFT_in_plane_slice0',
            peak=ev_to_hz(EV),
            plane='Exy',
            save_path=os.path.join('field_plots_' + name, f'{EV:.2f}eV_IN.png')
        )

h_axis(323, 403) v_axis(323, 403) cmap(323, 403)
  Saved: 5.05eV_IN.png
h_axis(323, 403) v_axis(323, 403) cmap(323, 403)
  Saved: 10.14eV_IN.png


In [27]:
EV_Efield_offset = [
    # 8.37, 15.99        # D100L100
    8.45, 16.01  # Th100 D100 
    # 10.17, 15.45       # D100 L25
    # 10.13, 5.398,           # D200L100
    # 10.23, 3.23, 18.33            # D200 L25

]

for EV in EV_Efield_offset:
    process_single_monitor(
            monitor_name='DFT_bottom_plane_slice14',
            peak=ev_to_hz(EV),
            plane='Exy',
            save_path=os.path.join('field_plots_' + name, f'{EV:.2f}eV_BOT.png')
        )

(163, 243) (163, 243) (163, 243)
  Saved: 8.45eV_BOT.png
(163, 243) (163, 243) (163, 243)
  Saved: 16.01eV_BOT.png


In [15]:
# Task
EV_Efield = 8.42
peak_freq = ev_to_hz(EV_Efield)

# Define all plotting tasks
tasks = [
    # ('DFT_in_plane_slice0',       'Exy', f'{EV_Efield:.2f}eV_IN_0.png'),
    # ('DFT_in_plane_slice4.5',     'Exy', f'{EV_Efield:.2f}eV_IN_4.5offset.png'),
    # ('DFT_bottom_plane_slice0.5', 'Exy', f'{EV_Efield:.2f}eV_BOT_0.5.png'),
    # ('DFT_bottom_plane_slice14',  'Exy', f'{EV_Efield:.2f}eV_BOT_14offset.png'),
    ('DFT_out_plane_XZ',          'Exz', f'{EV_Efield:.2f}eV_OUT_XZ.png'),
    ('DFT_out_plane_XZoffset',    'Exz', f'{EV_Efield:.2f}eV_OUT_XZoffset.png'),
]

# Process each monitor separately
for i, (monitor, plane, filename) in enumerate(tasks, 1):
    print(f"[{i}/{len(tasks)}]", end=" ")
    process_single_monitor(
        monitor_name=monitor,
        peak=peak_freq,
        plane=plane,
        save_path=os.path.join('field_plots_' + name, filename)
    )

[1/2]   Saved: 17.56eV_OUT_XZ.png
[2/2]   Saved: 17.56eV_OUT_XZoffset.png
